In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from concurrent.futures import ThreadPoolExecutor
from IPython.display import display, Markdown
import numpy as np
import sqlite3, json
import pandas as pd
import duckdb
import time

from typing import Literal, Optional
from pydantic import BaseModel, Field

In [2]:
local_llm = ChatOpenAI(
            api_key="ai",
            model="openai/gpt-oss-20b",
            base_url="http://192.168.0.110:8000/v1",
        )

In [4]:
#vllm server alive check
res = local_llm.invoke("안녕, 반가워")
res

AIMessage(content='안녕하세요! 반갑습니다. 오늘 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 75, 'total_tokens': 134, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': None, 'id': 'chatcmpl-6a5a137e9f89446f8d79de51c0bc0210', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d6b6c-c60a-72c0-a4d4-4a4bd8f091ce-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 75, 'output_tokens': 59, 'total_tokens': 134, 'input_token_details': {}, 'output_token_details': {}})

In [29]:
def compress_ship_data_duckdb_further(db_path, table_name, min_time, max_time, min_lon=125.678, max_lon=131.229, max_lat=36.001):
    db_path = db_path.replace("\\", "/") 
    con = duckdb.connect(database=':memory:')
    con.execute("INSTALL sqlite; LOAD sqlite;")
    con.execute(f"ATTACH '{db_path}' AS sqlite_db (TYPE SQLITE);")

    query = f"""
    -- [STEP 1] Raw 데이터 로드 및 1차 트리거 (정수 변환 비교로 정밀도 확보)
    WITH raw_data AS (
        SELECT *,
            CAST(timestamp AS TIMESTAMP) as ts,
            CAST(longitude AS DOUBLE) as lon_val,
            CAST(latitude AS DOUBLE) as lat_val,
            CAST(course AS DOUBLE) as c_course,
            CAST(speed AS DOUBLE) as s_speed,
            row_number() OVER () as temp_row_idx
        FROM sqlite_db.{table_name}
        WHERE longitude >= {min_lon} AND longitude <= {max_lon} AND latitude <= {max_lat}
            AND timestamp BETWEEN '{min_time}' AND '{max_time}'
    ),
    ordered_data AS (
        SELECT *,
            LAG(lon_val) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_lon,
            LAG(lat_val) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_lat,
            LAG(c_course) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_course,
            LAG(s_speed) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_speed
        FROM raw_data
    ),
    diff_calc AS (
        SELECT *,
            CASE WHEN CAST(lon_val * 1000 AS BIGINT) != CAST(COALESCE(prev_lon, lon_val) * 1000 AS BIGINT) 
                   OR CAST(lat_val * 1000 AS BIGINT) != CAST(COALESCE(prev_lat, lat_val) * 1000 AS BIGINT) 
                 THEN 1 ELSE 0 END as pos_change,
            CASE 
                WHEN (c_course - COALESCE(prev_course, c_course)) > 180 THEN (c_course - COALESCE(prev_course, c_course)) - 360
                WHEN (c_course - COALESCE(prev_course, c_course)) < -180 THEN (c_course - COALESCE(prev_course, c_course)) + 360
                ELSE (c_course - COALESCE(prev_course, c_course))
            END as course_diff,
            (s_speed - COALESCE(prev_speed, s_speed)) as speed_diff
        FROM ordered_data
    ),
    event_logic AS (
        SELECT *,
            -- 경계값 오차 방지를 위해 0.000001 보정 (Pandas와의 일치성 향상)
            CASE WHEN pos_change = 1 OR (ABS(course_diff) >= 19.999999 AND s_speed >= 0.999999) OR ABS(speed_diff) >= 1.999999 THEN 1 ELSE 0 END as event_trigger
        FROM diff_calc
    ),
    grouping_v1 AS (
        SELECT *,
            SUM(event_trigger) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as group_id_v1
        FROM event_logic
    ),
    summarized_v1 AS (
        -- [1차 압축] ANY_VALUE 대신 FIRST를 사용하여 Pandas .first()와 100% 일치화
        SELECT 
            ShipName, group_id_v1,
            FIRST(mmsi) as mmsi, FIRST(higher_types) as higher_types, FIRST(radius) as radius,
            MIN(ts) as start_time, MAX(ts) as end_time,
            FIRST(lon_val) as lon, FIRST(lat_val) as lat,
            FIRST(c_course) as first_course, AVG(s_speed) as avg_speed,
            FIRST(course_diff) as turn_val, FIRST(speed_diff) as accel_val
        FROM grouping_v1 
        GROUP BY ShipName, group_id_v1
    ),
    -- [STEP 2] 상태 판별 (부동소수점 오차 차단)
    status_calc AS (
        SELECT *,
            avg_speed - COALESCE(LAG(avg_speed) OVER (PARTITION BY ShipName ORDER BY start_time, group_id_v1), avg_speed) as group_speed_diff
        FROM summarized_v1
    ),
    status_final AS (
        SELECT *,
            CASE 
                -- 1.0, 20.0 등의 경계값을 소수점 8자리에서 반올림 후 비교하여 Pandas와 일치시킴
                WHEN ROUND(avg_speed, 8) < 1.0 THEN '정박/대기'
                ELSE TRIM(CONCAT_WS(' ',
                    CASE WHEN ROUND(ABS(turn_val), 8) >= 20.0 AND ROUND(avg_speed, 8) >= 1.0 
                         THEN (CASE WHEN turn_val > 0 THEN '우선회' ELSE '좌선회' END) || '(' || ROUND(ABS(turn_val), 1) || '°)' ELSE '' END,
                    CASE WHEN ROUND(group_speed_diff, 8) >= 2.0 THEN '가속' 
                         WHEN ROUND(group_speed_diff, 8) < -2.0 THEN '감속' ELSE '' END,
                    CASE WHEN ROUND(avg_speed, 8) >= 5.0 THEN '이동/통과' ELSE '저속 운항' END
                ))
            END as status
        FROM status_calc
    ),
    -- [STEP 3] 2차 압축
    v2_trigger AS (
        SELECT *,
            CASE WHEN status != COALESCE(LAG(status) OVER (PARTITION BY ShipName ORDER BY start_time, group_id_v1), status) 
                 THEN 1 ELSE 0 END as status_change
        FROM status_final
    ),
    v2_grouping AS (
        SELECT *,
            SUM(status_change) OVER (PARTITION BY ShipName ORDER BY start_time, group_id_v1 ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as group_id
        FROM v2_trigger
    )
    -- [STEP 4] 최종 요약 (집계 방식 일치)
    SELECT 
        ShipName, group_id,
        FIRST(mmsi) as mmsi, FIRST(higher_types) as higher_types, FIRST(radius) as radius,
        MIN(start_time) as start_time, MAX(end_time) as end_time,
        FIRST(lon) as lon, FIRST(lat) as lat,
        FIRST(first_course) as first_course,
        AVG(avg_speed) as avg_speed,
        FIRST(turn_val) as turn_val,
        FIRST(accel_val) as accel_val,
        FIRST(status) as status
    FROM v2_grouping
    GROUP BY ShipName, group_id
    ORDER BY ShipName, start_time
    """

    df_result = con.execute(query).df()
    con.execute("DETACH sqlite_db;")
    return df_result


In [22]:
ais_prompt_template ="""
    # Role
    당신은 대한민국 주변 선박운행을 관제하는 베테랑 해상 관제사(VTS Operator)이자 선박 항적 분석 전문가 입니다. 
    제공된 요약된 항적 데이터(Summarized Trajectory)를 바탕으로 선박의 이동 패턴과 주요 이벤트를 전문적인 자연어로 묘사해주세요.
    
    #Input Data
    1. Summarized Trajectory
    {trajectory_data}
    2. Current Weather data
    {weather_data}
    
    # Guidelines
    1. 시간 순서대로 전체적인 항해 흐름을 요약하고 선박이름을 명시해주세요.
    2. 요약할 때 기준은 "status"필드를 기준으로 하되 start_time과 end_time을 참고하세요. 
    3. trajectory_data는 ShipName이 같은 데이터끼리 리스트로 한번 더 묶여있으니 서로 ShipName이 서로 다른 배끼리 혼용하지 않도록 유의하세요.
    4. 첫번째 ShipName에 해당하는 현재날씨는 weather_data에서 첫번째 날씨 데이터를, 두번째 Shipnam에 해당하는 현재 날씨는 weather_data에서 두번째 날씨 데이터를 참고하세요.
    5. 전문적인 해상 관제 용어를 사용해도 되지만, 가독성이 좋게 작성하세요.
    6. 날씨데이터에 대해서 관제사가 고려하고 참고해야 할 사항에 대해 설명해주세요
    
    #Output Format
    1. 전체적인 항해 패턴을 간단하게 요약; 관제사가 관심을 가져야 할 특이 사항이 있을 시 간단하게 언급
    2. 현재 날씨에 대해 간단히 요약; 관제사가 관심을 가져야 할 날씨의 특이사항(풍향, 풍속 유의파고 등)이 있을 시 간단하게 언급 및 조치사항 설명
    3. 1, 2 항목을 종합적으로 정리해서 묘사
    """

each_ship_prompt = ChatPromptTemplate.from_template(ais_prompt_template)

In [24]:
summary_trajectory_template ="""
    #Input Data
    {all_analyses}
    
    # Guidelines
    1. Input Data의 정보를 종합해서 전체 배에 대한 내용을 종합적으로 요약해주세요
    2. 요약 시 각각의 배의 이름을 명시하고 특이사항을 중점적으로 요약하세요 
    3. 수치는 바뀌지 않도록 정확하게 참고해주세요
    4. 반복적인 내용은 짧게 요약하세요
    """

summary_trajectory_prompt = ChatPromptTemplate.from_template(summary_trajectory_template)

In [25]:
#병렬로 처리
analyze_chain = each_ship_prompt | local_llm | StrOutputParser()

# --- 1. 개별 분석 함수 (스레드에서 실행될 단위) ---
def analyze_single_ship(data):
    # data는 {'idx': i, 'trajectory': t, 'weather': w} 형태
    res = analyze_chain.invoke({
        "trajectory_data": data['trajectory'],
        "weather_data": data['weather']
    })
    return f"[ship_{data['idx']+1} 분석 결과]\n{res}"

# --- 2. 병렬 처리를 수행하는 래퍼 함수 ---
def parallel_wrapper(inputs):
    # inputs: {'tracks': [...], 'weathers': [...]}
    tracks = inputs['tracks']
    weathers = inputs['weathers']
    
    tasks = [
        {'idx': i, 'trajectory': tracks[i], 'weather': weathers[i]}
        for i in range(len(tracks))
    ]
    
    with ThreadPoolExecutor(max_workers=100) as executor:
        results_list = list(executor.map(analyze_single_ship, tasks))
    
    # 다음 체인(요약)을 위해 딕셔너리 형태로 반환
    return {
        "all_analyses": "\n\n".join(results_list),
        "ship_count": len(tracks)
    }

# --- 3. 하나의 거대한 체인으로 결합 ---
full_integrated_chain = (
    RunnableLambda(parallel_wrapper)  # 1단계: 병렬 분석 실행 및 결과 취합
    | summary_trajectory_prompt       # 2단계: 요약 프롬프트에 전달
    | local_llm                       # 3단계: 최종 요약 생성
    | StrOutputParser()               # 4단계: 텍스트만 추출
)

In [48]:
class RouteQuery(BaseModel):
    datasource: Literal["ship_info", "none"] = Field(
        ...,
        description="사용자 질문에 따라 'ship_info' 또는 'none'으로 라우팅합니다."
    )
    mmsi: Optional[list[int]] = Field(
        default=None,
        description="""datasource가 'ship_info'일 때만 해당 선박의 MMSI 번호를 추출하여 포함합니다.
        mmsi가 2개 이상일 경우 mmsi를 모두 포함하고 따옴표나 쌍따옴표 없이 int형으로 출력합니다 'none'일 경우 이 필드는 비워둡니다(null)."""
    )

structured_llm_router = local_llm.with_structured_output(RouteQuery)

system = """당신은 사용자 질문을 '선박 정보(ship_info)' 또는 '기타(none)'로 분류하는 전문 라우터입니다.
1. ship_info 선택 기준:
   - 사용자의 질문에 선박 이름(Ship Name) 또는 MMSI 번호가 구체적으로 포함된 경우.
   - 선박의 현재 위치, 상태, 항적 등을 묻는 질문인 경우.

2. none 선택 기준:
   - 질문에 특정 선박 이름이나 MMSI 번호가 없는 경우.
   - 인사, 일반적인 대화, 또는 해운/선박과 관련 없는 질문인 경우.
   - 선박에 대해 묻고 있지만, '어떤 선박'인지 식별할 수 있는 정보(이름/MMSI)가 전혀 없는 경우.

출력 규칙:
- datasource가 'ship_info'인 경우, 질문에서 추출한 MMSI 번호를 함께 반환하십시오.
- datasource가 'none'인 경우, MMSI 필드는 비워두십시오."""

route_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human","{question}"),
    ]
)

question_router = route_prompt | structured_llm_router

In [49]:
selected_data_source = question_router.invoke(
    {"question": "mmsi가 352001086와 431016257인 배가 현재 어디로 운항중이야??"}
)

print(selected_data_source)

datasource='ship_info' mmsi=[352001086, 431016257]


In [20]:
default_chain = local_llm | StrOutputParser()

In [26]:
retriever_chains = {
    'ship_info': full_integrated_chain,
    'none': default_chain
}

In [55]:
def query_routing_report(question):
    full_text = ""
    handle = display(Markdown(""), display_id=True)
    
    selected_data_source = question_router.invoke({"question": question})

    total_chain = retriever_chains[selected_data_source.datasource]
    mmsi_list = selected_data_source.mmsi

    testdb = "D:/AI_team/github/Vision_AI_RnD_team/projects/test_project/ais_weather/test.db"
    res_2 = compress_ship_data_duckdb_further(testdb, "AIS_category", '2022-12-01 00:00:00', '2022-12-01 23:59:59')

    res_2_sorted = res_2.sort_values(by=['ShipName', 'start_time'], ascending=True)

    filted_mmsi = res_2_sorted[res_2_sorted['mmsi'].isin(mmsi_list)].copy()
    
    ship_list = [
        group[1].to_dict(orient='records') 
        for group in filted_mmsi.groupby('ShipName', sort=False)
    ]

    #날씨DB에서 데이터 로드
    conn = sqlite3.connect('d:/AI_team/github/Vision_AI_RnD_team/projects/test_project/ais_weather/korea_weather.db', check_same_thread=False)
    weather_cursor = conn.cursor()
    
    # 4. 최종 DB 포맷 조립 (예: '2025-12-10 3')
    current_hour_str = "2025-12-10 3"
    
    query = """
        SELECT 
            b.지점명,
            b.latitude, 
            b.longitude, 
            w.*
        FROM 
            weather_buoy AS w
        JOIN 
            buoy_position AS b ON w.지점 = b.지점
        WHERE 
            w.일시 LIKE ?
    """
    
    # 3. 쿼리 실행
    search_param = f"{current_hour_str}:%"
    weather_cursor.execute(query, (search_param,))
    rows = weather_cursor.fetchall()
    
    # 4. 데이터 출력 및 처리
    if rows:
        # 2. Pandas 데이터프레임으로 로드
        col_names = [desc[0] for desc in weather_cursor.description]
        df = pd.DataFrame(rows, columns=col_names)
    
        target_cols = [0, 1, 2] + list(range(5, len(df.columns)))
        final_df = df.iloc[:, target_cols]

    #mmsi 두개 이상일 때 각각의 최근위치
    last_df = filted_mmsi.groupby('mmsi').tail(1)
    coords = final_df[['latitude', 'longitude']].values
    weather_l = []
    
    for _, ship_row in last_df.iterrows():
        # 1. 선박 현재 위치 및 기본 정보 추출
        mmsi = ship_row['mmsi']
        curr_pos = np.array([ship_row['lat'], ship_row['lon']])
        
        # 2. 거리 계산 및 가장 가까운 지점 인덱스 추출
        dist_seq = np.sum((coords - curr_pos)**2, axis=1)
        closest_i = np.argmin(dist_seq)
        
        # 3. 가장 가까운 지점의 정보를 가져와서 선박 정보와 합치기
        closest_node = final_df.iloc[closest_i].to_dict()
        weather_l.append(closest_node)
    
    res = total_chain.invoke({
        "tracks": ship_list,
        "weathers": weather_l
    })
    
    return res
    

In [54]:
routing_rep = query_routing_report("mmsi가 352001086와 636093035 배가 현재 어디로 운항중이야?")
print(routing_rep)

[{'지점명': '통영', 'latitude': 34.3917, 'longitude': 128.225, '풍속(m/s)': 4.4, '풍향(deg)': 4.0, 'GUST풍속(m/s)': 6.0, '현지기압(hPa)': 1027.4, '습도(%)': 53, '기온(°C)': 9.3, '수온(°C)': 19.2, '최대파고(m)': 0.5, '유의파고(m)': 0.4, '평균파고(m)': 0.2, '파주기(sec)': 7.1, '파향(deg)': 61}, {'지점명': '울산', 'latitude': 35.3453, 'longitude': 129.8414, '풍속(m/s)': 5.9, '풍향(deg)': 337.0, 'GUST풍속(m/s)': 8.2, '현지기압(hPa)': 1026.5, '습도(%)': 53, '기온(°C)': 8.8, '수온(°C)': 19.0, '최대파고(m)': 1.5, '유의파고(m)': 0.9, '평균파고(m)': 0.6, '파주기(sec)': 7.1, '파향(deg)': 23}]
**전체 요약 – 두 선박 운항 및 현황**

| 선박 | 핵심 항해 패턴 | 특이·주목할 점 | 관제관찰 포인트 |
|------|----------------|---------------|----------------|
| **AGS‑신천지** | - 00:00‑14:10 정박/대기 (평균속도 0.008 kn) <br> - 14:15 북‑서방 248°로 **62° 우선회** 후 가속 2.6 kn → 14:18 246° 우선회, 3.3→7.2 kn 가속 <br> - 14:25‑14:30 11 kn 가속, 19° 좌선회, 12 kn 초반 <br> - 14:30‑16:30 12 kn(≈20 m·s⁻¹) 지속 운항 <br> - 16:35‑20:00 ~11.3 kn 서남방 이동, 20:05‑21:00 315° 급진선·감속 (≈7.7 kn) <br> - 21:02‑21:18 8–10 kn 지속, 21:20‑22:46 8–11 kn(22:05‑22:46 6.5 kn 